#### Important Note: Almost all the code was initially written in the python files, here we have extracted only the main core functionality for every task. Because of this for task 5 I am using our custom word2vec model that was trained on more data and 300 dimensions, and the code in this notebook just represents the structure, with smaller parameters and data

# NLP Features Extraction Notebook

In [1]:
import json
import re
from typing import Any

import numpy as np
import pandas as pd
import pymorphy2
import spacy

# For Custom Embeddings
from gensim.models import Word2Vec
from googletrans import Translator

# For Pretrained Embeddings
from sentence_transformers import SentenceTransformer

# For TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# For Sentiment Analysis
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load full data for proper TF-IDF calculations
data = pd.read_excel("Data_Task4_initial.xlsx")
df = data.copy()

C:\Users\Alex\Documents\GitHub\2025-26a-fai2-adsai-OleksiiKrasnoshtanov240247\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Task 1. POS Extraction

In [2]:
nlp = spacy.load("ru_core_news_sm")


def extract_pos_tags(text: str) -> list[tuple[str, str]]:
    if not text or pd.isna(text):
        return []

    doc = nlp(text.strip())
    pos_tags = []

    for token in doc:
        if not token.is_space and not token.is_punct:
            pos_tags.append((token.text, token.pos_))

    return pos_tags


print("Extracting POS tags...")
df["POS_Tags"] = df["Russian_Text"].apply(extract_pos_tags)

Extracting POS tags...


We have decided to use list of tuples as the output format for POS extraction because it will be easier to handle in the future, then just list of [word/POS, word/POS] if we will need so.

### Task 2. TF-IDF CALCULATION

In [3]:
def preprocess_text(text: str) -> str:
    if not text or pd.isna(text):
        return ""

    text = text.lower()
    text = re.sub(r"[^а-яё\s]", " ", text)
    words = text.split()
    words = [word.strip() for word in words if len(word.strip()) >= 2]

    return " ".join(words)


def group_sentences(sentences: list[str], sentences_per_doc: int = 5) -> list[str]:
    grouped_docs = []
    for i in range(0, len(sentences), sentences_per_doc):
        sentence_group = sentences[i : i + sentences_per_doc]
        valid_sentences = [s for s in sentence_group if s and pd.notna(s)]
        if valid_sentences:
            combined_doc = " ".join(valid_sentences)
            grouped_docs.append(combined_doc)
    return grouped_docs


print("Calculating TF-IDF with sentence grouping...")
# Group sentences into larger documents (5 sentences per document)
sentences_per_doc = 5
grouped_documents = group_sentences(df["Russian_Text"].tolist(), sentences_per_doc)
processed_docs = [preprocess_text(doc) for doc in grouped_documents]
processed_docs = [doc for doc in processed_docs if doc.strip()]

print(f"Created {len(grouped_documents)} grouped documents from {len(df)} sentences")

if processed_docs:
    vectorizer = TfidfVectorizer(
        max_features=1000, token_pattern=r"\b[а-яё]+\b", min_df=1, max_df=0.95
    )

    tfidf_matrix = vectorizer.fit_transform(processed_docs)
    feature_names = vectorizer.get_feature_names_out().tolist()

    # Map TF-IDF back to individual sentences
    tfidf_results = []
    for i in range(len(df)):
        doc_index = i // sentences_per_doc  # Which grouped document this sentence belongs to

        if doc_index < tfidf_matrix.shape[0]:
            row = tfidf_matrix.getrow(doc_index)
            term_weights = {}
            for j, weight in zip(row.indices, row.data):
                if weight >= 0.001:
                    term_weights[feature_names[j]] = float(weight)
            tfidf_results.append(term_weights)
        else:
            tfidf_results.append({})

    df["TF_IDF"] = tfidf_results
    df["TF_IDF_Document_Group"] = [i // sentences_per_doc for i in range(len(df))]
else:
    df["TF_IDF"] = [{}] * len(df)
    df["TF_IDF_Document_Group"] = [0] * len(df)

Calculating TF-IDF with sentence grouping...
Created 149 grouped documents from 742 sentences


We have tried to use just basic math formula, but since we are not restricted, we decided to first of all group sentences into five row documents and then use vectorizer, because it was faster. We used a sparse vector and dictionary format to save values for us to have possibility as people to understand the output and for a machine to use it at the same time

### Task 3. SENTIMENT ANALYSIS

In [4]:
vader = SentimentIntensityAnalyzer()
translator = Translator()


def analyze_sentiment(text: str) -> float:
    if not text or not isinstance(text, str):
        return 0.0

    try:
        translated = translator.translate(text, src="ru", dest="en")
        english_text = translated.text
        scores = vader.polarity_scores(english_text)
        return scores["compound"]
    except:
        return 0.0


print("Analyzing sentiment...")
df["Sentiment_Score"] = df["Russian_Text"].apply(analyze_sentiment)

Analyzing sentiment...


We have tried to use different options for sentiment analysis without translation, like spacy model or dostoyevsky specifically trained model, but they all were either incompatible with our environment or yielding really strange results of >90% zeros so because we have used TV show data for this task we understood that we do not have much sarcasm or indirectness in the current speeech, so translating and using Vader seemed like a great option, which gave consistent results among all data

### Task 4. PRETRAINED WORD EMBEDDINGS

In [5]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")


def sentence_to_vector(sentence: str) -> np.ndarray:
    if pd.isna(sentence) or not sentence:
        return np.zeros(384)  # Model dimension

    try:
        sentence_str = str(sentence).strip()
        if not sentence_str:
            return np.zeros(384)

        embedding = sentence_model.encode([sentence_str])[0]
        return embedding
    except:
        return np.zeros(384)


print("Generating pretrained embeddings...")
embeddings = []
for text in df["Russian_Text"]:
    embedding = sentence_to_vector(text)
    embeddings.append(embedding)

df["Word_Embeddings"] = embeddings
df["Embedding_Dimension"] = [len(emb) for emb in embeddings]

Generating pretrained embeddings...


Just found multilingual transformer embedding model that had russian as one of the training languages, evaluation is in the nextt block

#### 4.1 PRETRAINED MODEL EVALUATION

In [6]:
print("Evaluating pretrained SentenceTransformer model...")

# Model information
print("\nModel Information:")
print("  Model name: all-MiniLM-L6-v2")
print(f"  Embedding dimension: {embeddings[0].shape[0]}")
print("  Model type: Multilingual sentence transformer")

# Test word similarity (treat words as single-word sentences)
test_words = ["человек", "дом", "работа", "время", "жизнь", "мир", "любовь"]
print("\nWord similarity examples (as single-word sentences):")

word_embeddings = {}
for word in test_words:
    word_embeddings[word] = sentence_model.encode([word])[0]

# Calculate similarities between test words
for i, word1 in enumerate(test_words[:4]):
    similarities = []
    for word2 in test_words:
        if word1 != word2:
            sim = cosine_similarity([word_embeddings[word1]], [word_embeddings[word2]])[0][0]
            similarities.append((word2, sim))

    # Sort by similarity and show top 3
    similarities.sort(key=lambda x: x[1], reverse=True)
    similar_words = [f"{w}({s:.3f})" for w, s in similarities[:3]]
    print(f"  {word1}: {similar_words}")

print("\nNote: SentenceTransformer optimized for sentences, not individual words")

# Test sentence similarity with Russian examples
test_sentences = [
    "Я люблю свою семью",
    "Моя семья очень важна для меня",
    "Работа занимает много времени",
    "Сегодня хорошая погода",
]

print("\nSentence similarity examples:")
test_embeddings = [sentence_model.encode([sent])[0] for sent in test_sentences]

from sklearn.metrics.pairwise import cosine_similarity

for i, sent1 in enumerate(test_sentences):
    for j, sent2 in enumerate(test_sentences):
        if i < j:
            sim = cosine_similarity([test_embeddings[i]], [test_embeddings[j]])[0][0]
            print(f"  '{sent1[:20]}...' vs '{sent2[:20]}...': {sim:.3f}")

# Analyze embeddings statistics
all_embeddings = np.array(embeddings)
print("\nEmbedding Statistics:")
print(
    f"  Successfully generated: {len([e for e in embeddings if not np.allclose(e, 0)])}/{len(embeddings)} sentences"
)
print(f"  Average embedding norm: {np.mean([np.linalg.norm(e) for e in embeddings]):.3f}")
print(f"  Embedding range: [{all_embeddings.min():.3f}, {all_embeddings.max():.3f}]")

# Unknown words handling explanation
print("\nUnknown Words Handling:")
print("  SentenceTransformer uses subword tokenization")
print("  No truly 'unknown' words - all text can be processed")
print("  Multilingual training includes Russian support")

Evaluating pretrained SentenceTransformer model...

Model Information:
  Model name: all-MiniLM-L6-v2
  Embedding dimension: 384
  Model type: Multilingual sentence transformer

Word similarity examples (as single-word sentences):
  человек: ['жизнь(0.543)', 'работа(0.525)', 'любовь(0.509)']
  дом: ['время(0.545)', 'работа(0.514)', 'мир(0.454)']
  работа: ['человек(0.525)', 'любовь(0.524)', 'дом(0.514)']
  время: ['мир(0.620)', 'дом(0.545)', 'работа(0.464)']

Note: SentenceTransformer optimized for sentences, not individual words

Sentence similarity examples:
  'Я люблю свою семью...' vs 'Моя семья очень важн...': 0.564
  'Я люблю свою семью...' vs 'Работа занимает мног...': 0.461
  'Я люблю свою семью...' vs 'Сегодня хорошая пого...': 0.382
  'Моя семья очень важн...' vs 'Работа занимает мног...': 0.569
  'Моя семья очень важн...' vs 'Сегодня хорошая пого...': 0.485
  'Работа занимает мног...' vs 'Сегодня хорошая пого...': 0.471

Embedding Statistics:
  Successfully generated: 742/74

Pretty nice and meaningful results

### Task 5. CUSTOM WORD EMBEDDING MODEL

In [7]:
morph = pymorphy2.MorphAnalyzer()
stop_words = {"и", "в", "во", "не", "что", "он", "на", "я", "с", "со", "как", "а", "то", "все"}


def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = re.sub(r"[^\w\s\-]", " ", str(text))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()


def process_text(text: str) -> list[str]:
    cleaned = clean_text(text)
    words = cleaned.split()

    processed_words = []
    for word in words:
        if len(word) > 2 and word not in stop_words:
            parsed = morph.parse(word)[0]
            lemma = parsed.normal_form
            if lemma and len(lemma) > 2:
                processed_words.append(lemma)

    return processed_words


# Create simple corpus from available text (demonstration)
print("Training simple Word2Vec model (for demonstration)...")
sentences = []
for text in df["Russian_Text"]:
    tokens = process_text(text)
    if len(tokens) > 2:
        sentences.append(tokens)

if sentences:
    # Train simple Word2Vec model (demonstration only)
    demo_model = Word2Vec(
        sentences=sentences, vector_size=100, window=5, min_count=1, workers=2, epochs=10
    )
    print(f"Demo model trained: vocab={len(demo_model.wv)}, dim={demo_model.wv.vector_size}")

Training simple Word2Vec model (for demonstration)...
Demo model trained: vocab=881, dim=100


This is exact structure we used to train our model, but in the file we have used 500 wikipedia articles as corpus and 300 dimensions, so it yields much better results as this small demo model trained on the spot

#### 5.1 LOAD INITIAL PRETRAINED CUSTOM MODEL

In [8]:
print("Loading pre-trained Russian Word2Vec model...")
MODEL_PATH = r"C:\Users\Alex\Documents\GitHub\2025-26a-fai2-adsai-OleksiiKrasnoshtanov240247\DataLabs\Task4\russian_word2vec.model"

try:
    custom_model = Word2Vec.load(MODEL_PATH)
    print(f"Model loaded: vocab={len(custom_model.wv)}, dim={custom_model.wv.vector_size}")
    model_loaded = True
except:
    print("Could not load pre-trained model, using demo model")
    custom_model = demo_model if "demo_model" in locals() else None
    model_loaded = False

Loading pre-trained Russian Word2Vec model...
Model loaded: vocab=17355, dim=300


#### Hyperparameter used Rationale

**vector_size=300**: Provides enough dimensions to capture Russian's morphological relationships and vocabulary without excessive computational cost.

**window=5**: Balances local syntactic context with broader semantic relationships.

**min_count=5**: Filters out typos and extremely rare word forms while preserving the linguistic richness

**sg=1 (Skip-gram)**: Better at handling infrequent words compared to CBOW, which is important for Russian literature and formal texts that contain many semantically valuable but low-frequency terms.

**epochs=100**: Ensures proper convergence for the relatively small dataset size (~500 Wikipedia articles). Smaller corpora need more training iterations to develop stable word representations.

**workers=4**: Standard parallelization that most machines can handle efficiently without memory issues.

#### 5.2 MODEL EVALUATION

In [9]:
if custom_model and model_loaded:
    print("Evaluating custom Word2Vec model...")

    # Test word similarities
    test_words = ["человек", "дом", "работа", "время", "жизнь", "мир", "любовь"]
    print("\nWord similarity examples:")
    for word in test_words:
        if word in custom_model.wv:
            similar = custom_model.wv.most_similar(word, topn=3)
            print(f"  {word}: {[s[0] for s in similar]}")
        else:
            print(f"  {word}: not in vocabulary")

    # Model statistics
    vocab_size = len(custom_model.wv)
    vector_dim = custom_model.wv.vector_size
    print("\nModel Statistics:")
    print(f"  Vocabulary size: {vocab_size}")
    print(f"  Vector dimension: {vector_dim}")

    # Coverage analysis on our data
    total_words = 0
    covered_words = 0
    for text in df["Russian_Text"]:
        words = process_text(text)
        total_words += len(words)
        for word in words:
            if word in custom_model.wv:
                covered_words += 1

    coverage = covered_words / total_words if total_words > 0 else 0
    print(f"  Coverage on our data: {coverage:.1%} ({covered_words}/{total_words} words)")
else:
    print("Skipping evaluation - no model available")

Evaluating custom Word2Vec model...

Word similarity examples:
  человек: ['футляр', 'инвалидность', 'интернет-пользователь']
  дом: ['пушкинский', 'доходный', 'наб']
  работа: ['винокур', 'щерба', 'торн']
  время: ['настоящий', 'антидребезг', 'бенедиктовый']
  жизнь: ['футляр', 'наслаждение', 'самгин']
  мир: ['четырёхмоторный', 'материк', 'наср']
  любовь: ['митин', 'переживание', 'привить']

Model Statistics:
  Vocabulary size: 17355
  Vector dimension: 300
  Coverage on our data: 77.2% (2652/3434 words)


The coverage is decent as well as the similarity between words, but we can see that some words are not covered by the model, which is not surprising as we have used only 500 articles from wikipedia, but we can see that the model is able to capture some of the morphological relationships, which is very important for our task, but it still is not perfect, and while really great for the word 'дом', for 'человек' it is not that great

#### 5.3 GENERATE CUSTOM EMBEDDINGS

In [10]:
def create_custom_embedding(text: str) -> np.ndarray:
    if not custom_model:
        return np.zeros(100)

    words = process_text(text)
    if not words:
        return np.zeros(custom_model.wv.vector_size)

    vectors = []
    for word in words:
        if word in custom_model.wv:
            vectors.append(custom_model.wv[word])

    if not vectors:
        return np.zeros(custom_model.wv.vector_size)

    return np.mean(vectors, axis=0)


print("Generating custom embeddings...")
df["Custom_Embeddings"] = df["Russian_Text"].apply(create_custom_embedding)

Generating custom embeddings...


### Task 6. NAMED ENTITY RECOGNITION

For custom NLP feature I have decided to use NER because our TV show that we have chosen as a test set is a travel show, and people there tell a lot of the names of cities, food dishes, mountains, etc. So for emotion detection they are not that important, so having this information may help reducing the impact of unnecessary word in the future

In [11]:
def extract_entities(text: str) -> dict[str, Any]:
    if not text or pd.isna(text):
        return {
            "entities": [],
            "entity_counts": {},
            "person_names": [],
            "locations": [],
            "organizations": [],
        }

    doc = nlp(str(text))

    entities = []
    person_names = []
    locations = []
    organizations = []
    entity_counts = {}

    for ent in doc.ents:
        entity_info = {"text": ent.text, "label": ent.label_}
        entities.append(entity_info)

        if ent.label_ == "PER":
            person_names.append(ent.text)
        elif ent.label_ in ["LOC", "GPE"]:
            locations.append(ent.text)
        elif ent.label_ == "ORG":
            organizations.append(ent.text)

        if ent.label_ not in entity_counts:
            entity_counts[ent.label_] = 0
        entity_counts[ent.label_] += 1

    return {
        "entities": entities,
        "entity_counts": entity_counts,
        "person_names": person_names,
        "locations": locations,
        "organizations": organizations,
    }


print("Extracting named entities...")
ner_results = df["Russian_Text"].apply(extract_entities)

df["NER_All_Entities"] = [json.dumps(result["entities"]) for result in ner_results]
df["NER_Persons"] = [", ".join(result["person_names"]) for result in ner_results]
df["NER_Locations"] = [", ".join(result["locations"]) for result in ner_results]
df["NER_Organizations"] = [", ".join(result["organizations"]) for result in ner_results]
df["NER_Entity_Count"] = [len(result["entities"]) for result in ner_results]

Extracting named entities...


### Results

In [12]:
df["Word_Embeddings_List"] = df["Word_Embeddings"].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)
df["Custom_Embeddings_List"] = df["Custom_Embeddings"].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)

# Create final output with selected columns
output_df = df[
    [
        "Russian_Text",
        "POS_Tags",
        "TF_IDF",
        "TF_IDF_Document_Group",
        "Sentiment_Score",
        "Word_Embeddings_List",
        "Custom_Embeddings_List",
        "NER_Persons",
        "NER_Locations",
        "NER_Organizations",
        "NER_Entity_Count",
    ]
].copy()

# Save to Excel (better for complex data than CSV)
output_df.to_excel("NLP_features.xlsx", index=False)
print(f"Results saved to NLP_features.xlsx with {len(output_df)} rows")

# Display summary
print("\nFeature extraction completed:")
print(f"- POS Tags: {sum(1 for tags in df['POS_Tags'] if tags)}/{len(df)} sentences")
print(
    f"- TF-IDF: {sum(1 for tfidf in df['TF_IDF'] if tfidf)}/{len(df)} sentences (grouped by {sentences_per_doc})"
)
print(f"- Sentiment: Average score = {df['Sentiment_Score'].mean():.3f}")
print(
    f"- Pretrained Embeddings: {df['Embedding_Dimension'].iloc[0]}D vectors (SentenceTransformer)"
)
print(
    f"- Custom Embeddings: {len(df['Custom_Embeddings'].iloc[0])}D vectors (pre-trained Word2Vec)"
)
print(f"- NER: {df['NER_Entity_Count'].sum()} total entities found")

# Show sample result
print("\nSample result for first row:")
print(f"Text: {df['Russian_Text'].iloc[0][:50]}...")
print(f"POS: {df['POS_Tags'].iloc[0][:3]}...")
print(
    f"TF-IDF group: {df['TF_IDF_Document_Group'].iloc[0]} (sentences {df['TF_IDF_Document_Group'].iloc[0] * sentences_per_doc}-{(df['TF_IDF_Document_Group'].iloc[0] + 1) * sentences_per_doc - 1})"
)
print(f"Sentiment: {df['Sentiment_Score'].iloc[0]:.3f}")
print(f"Entities: {df['NER_Entity_Count'].iloc[0]}")
print(f"Pretrained embedding dim: {len(df['Word_Embeddings'].iloc[0])}")
print(f"Custom embedding dim: {len(df['Custom_Embeddings'].iloc[0])}")

Results saved to NLP_features.xlsx with 742 rows

Feature extraction completed:
- POS Tags: 742/742 sentences
- TF-IDF: 742/742 sentences (grouped by 5)
- Sentiment: Average score = 0.030
- Pretrained Embeddings: 384D vectors (SentenceTransformer)
- Custom Embeddings: 300D vectors (pre-trained Word2Vec)
- NER: 125 total entities found

Sample result for first row:
Text: Мама нашла мужика из себя, который младше ее, на 1...
POS: [('Мама', 'NOUN'), ('нашла', 'VERB'), ('мужика', 'NOUN')]...
TF-IDF group: 0 (sentences 0-4)
Sentiment: -0.421
Entities: 0
Pretrained embedding dim: 384
Custom embedding dim: 300
